In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
#Self Attention

In [3]:
class SelfAttention(nn.Module):
    def __init__(self, d_model = 2, row_dim = 0, col_dim = 1):

        super(SelfAttention, self).__init__()

        self.W_q = nn.Linear(in_features = d_model,
                             out_features = d_model,
                             bias = False)

        self.W_k = nn.Linear(in_features = d_model,
                             out_features = d_model,
                             bias = False)

        self.W_v = nn.Linear(in_features = d_model,
                             out_features = d_model,
                             bias = False)

        self.row_dim = row_dim
        self.col_dim = col_dim


    def forward(self, token_encodings):

        q = self.W_q(token_encodings)
        k = self.W_k(token_encodings)
        v = self.W_v(token_encodings)

        sims = torch.matmul(q, k.transpose(dim0 = self.row_dim,
                                           dim1 = self.col_dim))

        scaled_sims = sims / torch.tensor(k.size(self.col_dim)**0.5)
        attention_percents = F.softmax(scaled_sims, dim= self.col_dim)

        attention_scores = torch.matmul(attention_percents, v)

        return attention_scores


In [4]:
encodings_matrix = torch.tensor([[1.16, 0.23],
                                 [0.57, 1.36],
                                 [4.41, -2.16]])

torch.manual_seed(42)

selfAttention = SelfAttention(d_model = 2,
                          row_dim = 0,
                          col_dim = 1)

selfAttention(encodings_matrix)



tensor([[1.0100, 1.0641],
        [0.2040, 0.7057],
        [3.4989, 2.2427]], grad_fn=<MmBackward0>)

In [5]:
selfAttention.W_q.weight.transpose(0,1)
selfAttention.W_k.weight.transpose(0,1)
selfAttention.W_v.weight.transpose(0,1)

tensor([[ 0.6233,  0.6146],
        [-0.5188,  0.1323]], grad_fn=<TransposeBackward0>)

### Self Attention


In [9]:
class MaskedSelfAttention(nn.Module):

    def __init__(self, d_model = 2, row_dim = 0, col_dim = 1):

        super(MaskedSelfAttention, self).__init__()

        self.W_q = nn.Linear(in_features = d_model,
                             out_features = d_model,
                             bias = False)
        self.W_k = nn.Linear(in_features = d_model,
                             out_features = d_model,
                             bias = False)
        self.W_v = nn.Linear(in_features = d_model,
                             out_features = d_model,
                                bias = False)

        self.row_dim = row_dim
        self.col_dim = col_dim

    def forward(self, token_encodings, mask = None):
        q = self.W_q(token_encodings)
        k = self.W_k(token_encodings)
        v = self.W_v(token_encodings)

        sims = torch.matmul(q, k.transpose(dim0 = self.row_dim,
                                           dim1 = self.col_dim))

        scaled_sims = sims / torch.tensor(k.size(self.col_dim)**0.5)

        if mask is not None:
            scaled_sims = scaled_sims.masked_fill(mask = mask, value = -1e9)

        attention_percents = F.softmax(scaled_sims, dim= self.col_dim)

        attention_scores = torch.matmul(attention_percents, v)

        return attention_scores

In [ ]:
encodings_matrix = torch.tensor([[1.16, 0.23],
                                 [0.57, 1.36],
                                 [4.41, -2.16]])

torch.manual_seed(42)
maskedSelfAttention = MaskedSelfAttention(d_model = 2,
                                          row_dim = 0,
                                            col_dim = 1)

mask = torch.tril(torch.ones(3, 3))

tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])

In [11]:
maskedSelfAttention(encodings_matrix, mask = mask)

tensor([[ 0.6038,  0.7434],
        [-0.0062,  0.6072],
        [ 3.4989,  2.2427]], grad_fn=<MmBackward0>)

In [12]:
maskedSelfAttention.W_q.weight.transpose(0,1)

tensor([[ 0.5406, -0.1657],
        [ 0.5869,  0.6496]], grad_fn=<TransposeBackward0>)

# Implement Encoder-Decoder Attention and Multi-Head Attention

In [13]:
class Attention(nn.Module):
    def __init__(self, d_model = 2, row_dim = 0, col_dim = 1):

        super(Attention, self).__init__()

        self.W_q = nn.Linear(in_features = d_model,
                             out_features = d_model,
                             bias = False)

        self.W_k = nn.Linear(in_features = d_model,
                             out_features = d_model,
                             bias = False)

        self.W_v = nn.Linear(in_features = d_model,
                             out_features = d_model,
                             bias = False)

        self.row_dim = row_dim
        self.col_dim = col_dim


    def forward(self, encodings_for_q, encodings_for_k, encodings_for_v, mask = None):

        q = self.W_q(encodings_for_q)
        k = self.W_k(encodings_for_k)
        v = self.W_v(encodings_for_v)

        sims = torch.matmul(q, k.transpose(dim0 = self.row_dim,
                                           dim1 = self.col_dim))

        scaled_sims = sims / torch.tensor(k.size(self.col_dim)**0.5)

        if mask is not None:
            scaled_sims = scaled_sims.masked_fill(mask = mask, value = -1e9)

        attention_percents = F.softmax(scaled_sims, dim= self.col_dim)

        attention_scores = torch.matmul(attention_percents, v)

        return attention_scores

In [14]:
encodings_for_q = torch.tensor([[1.16, 0.23],
                                    [0.57, 1.36],
                                    [4.41, -2.16]])

encodings_for_k = torch.tensor([[1.16, 0.23],
                                    [0.57, 1.36],
                                    [4.41, -2.16]])

encodings_for_v = torch.tensor([[1.16, 0.23],
                                    [0.57, 1.36],
                                    [4.41, -2.16]])

torch.manual_seed(42)

attention = Attention(d_model = 2,
                      row_dim = 0,
                        col_dim = 1)

attention(encodings_for_q, encodings_for_k, encodings_for_v)


tensor([[1.0100, 1.0641],
        [0.2040, 0.7057],
        [3.4989, 2.2427]], grad_fn=<MmBackward0>)

In [15]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model = 2, row_dim = 0, col_dim = 1, num_heads = 1):

        super(MultiHeadAttention, self).__init__()

        self.heads = nn.ModuleList([Attention(d_model = d_model,
                                              row_dim = row_dim,
                                              col_dim = col_dim) for _ in range(num_heads)])

        self.col_dim = col_dim

    def forward(self, encodings_for_q, encodings_for_k, encodings_for_v):
        return torch.cat([head(encodings_for_q, encodings_for_k, encodings_for_v) for head in self.heads], dim = self.col_dim)

In [16]:
torch.manual_seed(42)

multiHeadAttention = MultiHeadAttention(d_model = 2,
                                        row_dim = 0,
                                        col_dim = 1,
                                        num_heads = 1)

multiHeadAttention(encodings_for_q, encodings_for_k, encodings_for_v)

tensor([[1.0100, 1.0641],
        [0.2040, 0.7057],
        [3.4989, 2.2427]], grad_fn=<CatBackward0>)

In [17]:
torch.manual_seed(42)

multiHeadAttention = MultiHeadAttention(d_model = 2,
                                        row_dim = 0,
                                        col_dim = 1,
                                        num_heads = 2)

multiHeadAttention(encodings_for_q, encodings_for_k, encodings_for_v)

tensor([[ 1.0100,  1.0641, -0.7081, -0.8268],
        [ 0.2040,  0.7057, -0.7417, -0.9193],
        [ 3.4989,  2.2427, -0.7190, -0.8447]], grad_fn=<CatBackward0>)